##Setup and Dataset (with AMP)
We add torch.cuda.amp to the imports. The rest of this section is the same.

In [1]:
# Install libraries
!pip install -q albumentations

# Imports
import torch
import os
import random
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn # NEW: Import MobileNet
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from tqdm import tqdm
import torch.cuda.amp as amp # NEW: Import for Mixed-Precision

# --- Kaggle/Device Setup, Dataset, and Transform Functions (No Change) ---
# ... (Copy-paste all the setup, get_transform, PlateAlbumentationsDataset,
# and collate_fn_ensemble code from the previous step)
# === DIRECT KAGGLE CREDENTIALS ===
os.environ['KAGGLE_USERNAME'] = 'asd147'
os.environ['KAGGLE_KEY']     = '6a95e405001115800e2e18044513a965'
!kaggle datasets download -d andrewmvd/car-plate-detection
!unzip -q -o car-plate-detection.zip -d ./data
!rm -f car-plate-detection.zip
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
IMG_WIDTH, IMG_HEIGHT = 512, 512
def get_transform(train):
    if train:
        return A.Compose([A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75), A.GaussianBlur(blur_limit=(3, 5), p=0.3), A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
    else:
        return A.Compose([A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
class PlateAlbumentationsDataset(Dataset):
    def __init__(self, data_dir, image_files, width, height, transforms=None):
        self.transforms, self.data_dir, self.height, self.width, self.image_files = transforms, data_dir, height, width, image_files
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = cv2.cvtColor(cv2.imread(os.path.join(self.data_dir, 'images', img_name)), cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image, (self.width, self.height), interpolation=cv2.INTER_LANCZOS4)
        root = ET.parse(os.path.join(self.data_dir, 'annotations', img_name.replace('.png', '.xml'))).getroot()
        orig_w, orig_h = int(root.find('size/width').text), int(root.find('size/height').text)
        b = root.find('object/bndbox')
        boxes = [[(float(b.find(t).text)/s)*d for t,s,d in zip(['xmin','ymin','xmax','ymax'], [orig_w,orig_h,orig_w,orig_h], [self.width,self.height,self.width,self.height])]]
        target = {'boxes': boxes, 'labels': [1]}
        if self.transforms:
            augmented = self.transforms(image=image_resized, bboxes=target['boxes'], labels=target['labels'])
            image_tensor = augmented['image']
            target['boxes'] = torch.as_tensor(augmented['bboxes'], dtype=torch.float32) if augmented['bboxes'] else torch.zeros((0, 4), dtype=torch.float32)
            target['labels'] = torch.as_tensor(augmented['labels'], dtype=torch.int64)
        return image_resized, image_tensor, target
    def __len__(self): return len(self.image_files)
def collate_fn_ensemble(batch): return list(zip(*batch))

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/car-plate-detection
License(s): CC0-1.0
100% 203M/203M [00:14<00:00, 15.0MB/s]

Using device: cuda


##Speed-Optimized Training Loop
This is where the magic happens. We switch the model, create a 5-model ensemble, and wrap the training step with amp.autocast().

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def create_efficient_model(): # NEW function for MobileNet
    model = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
    return model

num_models = 5 # NEW: Increased ensemble size
ensemble_models = []
seeds = [42, 84, 126, 168, 210] # More seeds for more models
data_dir = './data'
all_image_files = sorted([f for f in os.listdir(os.path.join(data_dir, 'images')) if f.endswith('.png')])

for i in range(num_models):
    print(f"--- Training EFFICIENT Model {i+1}/{num_models} with Seed {seeds[i]} ---")
    set_seed(seeds[i])

    train_files, _ = train_test_split(all_image_files, test_size=0.2, random_state=seeds[i])
    train_dataset = PlateAlbumentationsDataset(data_dir, train_files, IMG_WIDTH, IMG_HEIGHT, get_transform(train=True))
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda b: tuple(zip(*b))[1:]) # Increased batch size is often possible

    model = create_efficient_model().to(device)

    num_epochs = 20
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=0.001, weight_decay=0.0005) # AdamW can converge faster
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.00001)

    scaler = amp.GradScaler() # NEW: Create a gradient scaler for AMP

    for epoch in range(num_epochs):
        model.train()
        for images, targets in tqdm(train_loader, desc=f"Model {i+1} Epoch {epoch+1}"):
            valid_batch = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt['boxes']) > 0]
            if not valid_batch: continue
            images_b, targets_b = zip(*valid_batch)
            images_b = list(img.to(device) for img in images_b)
            targets_b = [{k: v.to(device) for k, v in t.items()} for t in targets_b]

            # NEW: AMP training context
            with amp.autocast():
                loss_dict = model(images_b, targets_b)
                losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            scaler.scale(losses).backward() # Scale the loss
            scaler.step(optimizer)         # Scaler steps the optimizer
            scaler.update()                # Update the scale for next iteration

        lr_scheduler.step()

    ensemble_models.append(model)
    print(f"--- Finished Training EFFICIENT Model {i+1} ---")

--- Training EFFICIENT Model 1/5 with Seed 42 ---
Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 189MB/s]
/tmp/ipykernel_28534/807055559.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler() # NEW: Create a gradient scaler for AMP
Model 1 Epoch 1:   0%|          | 0/44 [00:00<?, ?it/s]/tmp/ipykernel_28534/807055559.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Model 1 Epoch 20: 100%|██████████| 44/44 [00:16<00:00,  2.73it/s]


--- Finished Training EFFICIENT Model 1 ---
--- Training EFFICIENT Model 2/5 with Seed 84 ---


Model 2 Epoch 20: 100%|██████████| 44/44 [00:16<00:00,  2.73it/s]


--- Finished Training EFFICIENT Model 2 ---
--- Training EFFICIENT Model 3/5 with Seed 126 ---


...

--- Finished Training EFFICIENT Model 3 ---
--- Training EFFICIENT Model 4/5 with Seed 168 ---


Model 4 Epoch 20: 100%|██████████| 44/44 [00:16<00:00,  2.74it/s]


--- Finished Training EFFICIENT Model 4 ---
--- Training EFFICIENT Model 5/5 with Seed 210 ---


Model 5 Epoch 20: 100%|██████████| 44/44 [00:16<00:00,  2.70it/s]

--- Finished Training EFFICIENT Model 5 ---


##Evaluation (Unchanged)
The evaluation logic is perfect as it is. It will take our new ensemble of 5 fast models and apply the same powerful TTA + Averaging technique.

In [3]:
# --- Evaluation Block (No Change) ---
# ... (Copy-paste the entire calculate_iou_detection and evaluation loop
# from the previous step)
def calculate_iou_detection(pred_boxes, true_boxes):
    if pred_boxes.shape[0] == 0 or true_boxes.shape[0] == 0: return torch.tensor(0.0)
    pred_boxes, true_boxes = pred_boxes.view(-1, 4), true_boxes.view(-1, 4)
    inter_xmin = torch.max(pred_boxes[:, 0].unsqueeze(1), true_boxes[:, 0])
    inter_ymin = torch.max(pred_boxes[:, 1].unsqueeze(1), true_boxes[:, 1])
    inter_xmax = torch.min(pred_boxes[:, 2].unsqueeze(1), true_boxes[:, 2])
    inter_ymax = torch.min(pred_boxes[:, 3].unsqueeze(1), true_boxes[:, 3])
    inter_area = torch.clamp(inter_xmax - inter_xmin, min=0) * torch.clamp(inter_ymax - inter_ymin, min=0)
    pred_area = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    true_area = (true_boxes[:, 2] - true_boxes[:, 0]) * (true_boxes[:, 3] - true_boxes[:, 1])
    union_area = pred_area.unsqueeze(1) + true_area - inter_area
    iou = inter_area / (union_area + 1e-6)
    return iou.max(dim=1)[0]
set_seed(42)
_, val_files = train_test_split(all_image_files, test_size=0.2, random_state=42)
val_dataset = PlateAlbumentationsDataset(data_dir, val_files, IMG_WIDTH, IMG_HEIGHT, get_transform(train=False))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn_ensemble)
for model in ensemble_models: model.eval()
total_iou, total_images = 0.0, 0
with torch.no_grad():
    for raw_images, images, targets in tqdm(val_loader, desc="Evaluating with Efficient Ensemble + TTA"):
        images = list(img.to(device) for img in images)
        for i in range(len(images)):
            ensemble_boxes = []
            for model in ensemble_models:
                with amp.autocast():
                    output_orig = model([images[i]])[0]
                    if len(output_orig['boxes']) > 0: ensemble_boxes.append(output_orig['boxes'][0].cpu())
                    img_flipped = torch.flip(images[i], dims=[2])
                    output_flipped = model([img_flipped])[0]
                    if len(output_flipped['boxes']) > 0:
                        box_flipped = output_flipped['boxes'][0].cpu()
                        ensemble_boxes.append(torch.tensor([IMG_WIDTH - box_flipped[2], box_flipped[1], IMG_WIDTH - box_flipped[0], box_flipped[3]]))
            if not ensemble_boxes: continue
            final_box = torch.stack(ensemble_boxes).mean(dim=0).unsqueeze(0)
            iou_score = calculate_iou_detection(final_box, targets[i]['boxes'].to('cpu'))
            total_iou += iou_score.sum().item()
        total_images += len(images)
avg_iou = total_iou / total_images
print(f"\nFINAL IoU with EFFICIENT Ensemble (5xMobileNetV3) + TTA: {avg_iou:.4f}")

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
Evaluating with Efficient Ensemble + TTA:   0%|          | 0/22 [00:00<?, ?it/s]/tmp/ipykernel_28534/501086066.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Evaluating with Efficient Ensemble + TTA: 100%|██████████| 22/22 [00:33<00:00,  1.54s/it]


FINAL IoU with EFFICIENT Ensemble (5xMobileNetV3) + TTA: 0.8018
